# Qdrant Backend — Production-Grade HNSW Vector Search

Medha 0.3.0 ships with `QdrantBackend`: the **flagship production backend** built
on [Qdrant](https://qdrant.tech), a purpose-built vector database with HNSW indexing,
vector quantization, and rich filtering capabilities.

Medha 0.3.0 supports ten pluggable backends via `Settings.backend_type`:

| `backend_type` | Backend | Extra deps | Persistence | Search | Best for |
|---|---|---|---|---|---|
| `"memory"` | `InMemoryBackend` | none | no | linear O(n) | **Default**; tests, CI, demos |
| `"qdrant"` | `QdrantBackend` | `medha-archai[qdrant]` | yes | **HNSW O(log n)** | **Production; docker/cloud + quantization** |
| `"pgvector"` | `PgVectorBackend` | `medha-archai[pgvector]` | yes | IVFFlat/HNSW | Teams already on PostgreSQL |
| `"elasticsearch"` | `ElasticsearchBackend` | `medha-archai[elasticsearch]` | yes | ANN | Teams on Elastic stack |
| `"vectorchord"` | `VectorChordBackend` | `medha-archai[vectorchord]` | yes | HNSW | High-throughput PostgreSQL |
| `"chroma"` | `ChromaBackend` | `medha-archai[chroma]` | yes | ANN | Lightweight local store |
| `"weaviate"` | `WeaviateBackend` | `medha-archai[weaviate]` | yes | ANN | Managed cloud search |
| `"redis"` | `RedisVectorBackend` | `medha-archai[redis]` | yes | HNSW/FLAT | Redis-stack teams |
| `"azure-search"` | `AzureSearchBackend` | `medha-archai[azure-search]` | yes | HNSW | Azure-native teams |
| `"lancedb"` | `LanceDBBackend` | `medha-archai[lancedb]` | yes | ANN | Zero-infra embedded |

This notebook covers:

1. **Imports & availability check**
2. **Memory mode** — zero-infra quick start (`:memory:`)
3. **Docker mode** — connect to a running Qdrant instance
4. **Full waterfall** — all cache tiers on Qdrant
5. **Template matching**
6. **HNSW tuning** — `hnsw_m`, `hnsw_ef_construct`
7. **Quantization** — scalar INT8 vs binary
8. **Quantization search params** — `rescore`, `oversampling`, `ignore`
9. **Hybrid storage** — `on_disk=True` with quantized vectors in RAM
10. **Multi-tenant isolation** — one collection per tenant
11. **TTL + `expire()`** — Qdrant `DatetimeRange` filter
12. **Cloud mode** — Qdrant Cloud (placeholder)
13. **Graceful degradation** — fallback to InMemoryBackend
14. **Cleanup** — `drop_collection()`

**Prerequisites:**
```bash
# 1. Install the Python package
pip install "medha-archai[qdrant,fastembed]"

# 2. (Optional) Run Qdrant via Docker for docker/cloud cells
docker run -d --name qdrant -p 6333:6333 -p 6334:6334 qdrant/qdrant

# 3. (Optional) Set the URL for docker-mode cells
export MEDHA_TEST_QDRANT_URL=http://localhost:6333
```


## Cell 1: Imports & Availability Check

In [ ]:
import asyncio
import os
import time

from medha import Medha, Settings, SearchStrategy
from medha.embeddings.fastembed_adapter import FastEmbedAdapter
from medha.types import QueryTemplate

try:
    from medha import QdrantBackend
    HAS_QDRANT = True
    print("qdrant-client is installed")
except ImportError:
    HAS_QDRANT = False
    print("qdrant-client not found — install with: pip install medha-archai[qdrant]")

QDRANT_URL = os.environ.get("MEDHA_TEST_QDRANT_URL", "")
if QDRANT_URL:
    print(f"MEDHA_TEST_QDRANT_URL is set: {QDRANT_URL}")
else:
    print("MEDHA_TEST_QDRANT_URL not set — docker/cloud cells will be skipped")
    print("  Start Qdrant: docker run -d --name qdrant -p 6333:6333 qdrant/qdrant")
    print("  Then: export MEDHA_TEST_QDRANT_URL=http://localhost:6333")

embedder = FastEmbedAdapter(model_name="BAAI/bge-small-en-v1.5", cache_dir="./.fastembed_cache")

CAN_RUN_DOCKER = HAS_QDRANT and bool(QDRANT_URL)

## Cell 2: Memory Mode — Zero-infra Quick Start

`qdrant_mode="memory"` runs an in-process Qdrant instance with no external service.
Data is lost when the process exits — ideal for unit tests and CI.

```python
Settings(backend_type="qdrant", qdrant_mode="memory")
```

In [ ]:
if not HAS_QDRANT:
    print("Skipping — qdrant-client not installed.")
else:
    settings = Settings(
        backend_type="qdrant",
        qdrant_mode="memory",
    )

    pairs = [
        ("How many users are there?",     "SELECT COUNT(*) FROM users"),
        ("List all active users",          "SELECT * FROM users WHERE status = 'active'"),
        ("Average salary by department",   "SELECT dept, AVG(salary) FROM employees GROUP BY dept"),
        ("Top 10 products by revenue",     "SELECT product_id, SUM(amount) FROM orders GROUP BY product_id ORDER BY SUM(amount) DESC LIMIT 10"),
        ("Orders placed today",            "SELECT COUNT(*) FROM orders WHERE DATE(created_at) = CURDATE()"),
    ]

    async with Medha("qdrant_demo", embedder=embedder, settings=settings) as medha:
        for q, sql in pairs:
            await medha.store(q, sql)

        print(f"Stored {len(pairs)} entries. Backend: {type(medha._backend).__name__}")
        print(f"Mode   : {settings.qdrant_mode}\n")

        hit = await medha.search("Total number of registered users")
        print(f"strategy   : {hit.strategy.value}")
        print(f"confidence : {hit.confidence:.4f}")
        print(f"query      : {hit.generated_query}")

## Cell 3: Docker Mode — Connect to a Running Qdrant Instance

`qdrant_mode="docker"` connects to a Qdrant server by URL or by `host:port`.

**Option A — `qdrant_url`** (preferred):
```python
Settings(backend_type="qdrant", qdrant_mode="docker", qdrant_url="http://localhost:6333")
```

**Option B — `qdrant_host` + `qdrant_port`** (default port 6333):
```python
Settings(backend_type="qdrant", qdrant_mode="docker", qdrant_host="qdrant.internal", qdrant_port=6333)
```

**Option C — environment variables**:
```bash
export MEDHA_BACKEND_TYPE=qdrant
export MEDHA_QDRANT_MODE=docker
export MEDHA_QDRANT_URL=http://localhost:6333
```

In [ ]:
if not CAN_RUN_DOCKER:
    print("Skipping — set MEDHA_TEST_QDRANT_URL to run docker-mode cells.")
else:
    settings = Settings(
        backend_type="qdrant",
        qdrant_mode="docker",
        qdrant_url=QDRANT_URL,
    )

    print(f"Connecting to Qdrant at: {QDRANT_URL}")

    async with Medha("qdrant_docker", embedder=embedder, settings=settings) as medha:
        await medha.store("How many users?", "SELECT COUNT(*) FROM users")
        count = await medha._backend.count("qdrant_docker")
        print(f"Connected. Collection count: {count}")

        hit = await medha.search("User count")
        print(f"strategy={hit.strategy.value}  query={hit.generated_query!r}")

        await medha._backend.drop_collection("qdrant_docker")
        print("Collection dropped.")

## Cell 4: Full Waterfall — All Cache Tiers on Qdrant

All four cache tiers work identically with `QdrantBackend`. Qdrant handles
vector search via HNSW, while exact and L1 tiers use payload filters.

| Tier | Mechanism | Qdrant operation |
|---|---|---|
| **L1** | In-memory Python dict | None — pure Python |
| **Exact** | Payload filter on `normalized_question` | `scroll` with `MatchValue` filter |
| **Semantic** | HNSW ANN | `query_points` with `score_threshold` |
| **Fuzzy** | Vector pre-filter + Levenshtein | `query_points` → `rapidfuzz` |

In [ ]:
if not HAS_QDRANT:
    print("Skipping — qdrant-client not installed.")
else:
    settings = Settings(
        backend_type="qdrant",
        qdrant_mode="memory",
    )

    async with Medha("qdrant_waterfall", embedder=embedder, settings=settings) as medha:
        pairs = [
            ("How many users are there?",        "SELECT COUNT(*) FROM users"),
            ("List all active users",             "SELECT * FROM users WHERE status = 'active'"),
            ("What is the average salary?",       "SELECT AVG(salary) FROM employees"),
            ("Show top 10 products by price",     "SELECT * FROM products ORDER BY price DESC LIMIT 10"),
            ("Count orders placed today",         "SELECT COUNT(*) FROM orders WHERE DATE(created_at) = CURDATE()"),
            ("Total revenue this month",          "SELECT SUM(amount) FROM orders WHERE MONTH(created_at) = MONTH(NOW())"),
        ]

        for q, sql in pairs:
            await medha.store(q, sql)

        test_queries = [
            ("How many users are there?",        "exact repeat (L1 on 2nd call)"),
            ("How many users are there?",        "L1 cache hit"),
            ("What's the total user count?",     "semantic paraphrase"),
            ("Show the priciest products",       "semantic paraphrase"),
            ("Something completely unrelated",   "should miss"),
        ]

        print(f"  {'Question':<45s}  {'Strategy':<22s}  {'Conf':>6s}  Note")
        print("  " + "-" * 90)

        for question, note in test_queries:
            t0 = time.perf_counter()
            hit = await medha.search(question)
            elapsed_ms = (time.perf_counter() - t0) * 1000
            conf = f"{hit.confidence:.3f}" if hit.confidence else "  n/a"
            print(f"  {question:<45s}  {hit.strategy.value:<22s}  {conf}  ({note}, {elapsed_ms:.1f}ms)")

## Cell 5: Template Matching

Templates are stored in a dedicated Qdrant collection (`__medha_templates_<name>`).
Payload indexes on `template_id` and `query_hash` are created for docker/cloud
modes — in memory mode they are skipped (Qdrant emits a warning and they have no effect).

In [ ]:
if not HAS_QDRANT:
    print("Skipping — qdrant-client not installed.")
else:
    settings = Settings(backend_type="qdrant", qdrant_mode="memory")

    async with Medha("qdrant_tmpl", embedder=embedder, settings=settings) as medha:
        templates = [
            QueryTemplate(
                intent="employees_by_dept",
                template_text="List employees in {department}",
                query_template="SELECT * FROM employees WHERE department = '{department}'",
                parameters=["department"],
                parameter_patterns={"department": r"\b(Engineering|Marketing|Sales|HR)\b"},
            ),
            QueryTemplate(
                intent="orders_by_status",
                template_text="Show {status} orders",
                query_template="SELECT * FROM orders WHERE status = '{status}'",
                parameters=["status"],
                parameter_patterns={"status": r"\b(pending|completed|cancelled|processing)\b"},
            ),
        ]

        await medha.load_templates(templates)
        print(f"Loaded {len(templates)} template(s)\n")

        for q in [
            "Show me employees in Engineering",
            "List staff in Sales",
            "Show pending orders",
            "Get all cancelled orders",
        ]:
            hit = await medha.search(q)
            print(f"  [{hit.strategy.value:16s}] {q!r}")
            if hit.generated_query:
                print(f"    → {hit.generated_query}")

## Cell 6: HNSW Tuning — `hnsw_m` and `hnsw_ef_construct`

HNSW (Hierarchical Navigable Small World) is Qdrant's ANN index. Two knobs control
its quality/speed tradeoff:

| Setting | Default | Effect |
|---|---|---|
| `hnsw_m` | 16 | Edges per node. Higher = better recall, more RAM |
| `hnsw_ef_construct` | 100 | Build-time search depth. Higher = better index quality, slower build |

**Rule of thumb:**
- Small collection (< 100k): defaults are fine
- Medium (100k – 1M): `hnsw_m=32`, `hnsw_ef_construct=200`
- Large (> 1M): `hnsw_m=64`, `hnsw_ef_construct=400`

Note: HNSW parameters are set at collection *creation* time and cannot be changed
without recreating the collection.

In [ ]:
if not HAS_QDRANT:
    print("Skipping — qdrant-client not installed.")
else:
    stored_q = "How many users are there?"
    search_q = "How many users are in the system?"
    sql      = "SELECT COUNT(*) FROM users"

    print(f"  {'hnsw_m':<8s}  {'ef_construct':<14s}  {'Strategy':<20s}  {'Confidence':>10s}")
    print("  " + "-" * 58)

    for m, ef in [(16, 100), (32, 200), (64, 400)]:
        settings = Settings(
            backend_type="qdrant",
            qdrant_mode="memory",
            hnsw_m=m,
            hnsw_ef_construct=ef,
        )
        async with Medha(f"hnsw_{m}_{ef}", embedder=embedder, settings=settings) as medha:
            await medha.store(stored_q, sql)
            hit = await medha.search(search_q)
            conf = f"{hit.confidence:.4f}" if hit.confidence else "   n/a"
            print(f"  {m:<8d}  {ef:<14d}  {hit.strategy.value:<20s}  {conf:>10s}")

    print("\nNote: differences are negligible on small collections — HNSW shines at scale.")

## Cell 7: Quantization — Scalar INT8 and Binary

Quantization reduces memory usage by compressing original `float32` vectors:

| Type | `quantization_type` | Condition | Memory reduction | Speed |
|---|---|---|---|---|
| **Scalar INT8** | `"scalar"` | any dimension | ~4x | fast |
| **Binary** | `"binary"` | dim ≥ 512 | ~32x | fastest |

Default: `enable_quantization=True`, `quantization_type="scalar"` for all dims;
binary is auto-selected when `quantization_type="binary"` **and** `dim >= 512`.

`quantization_always_ram=True` (default) keeps quantized vectors in RAM even when
`on_disk=True` — vectors used for search stay fast.

In [ ]:
if not HAS_QDRANT:
    print("Skipping — qdrant-client not installed.")
else:
    stored_q = "How many users are there?"
    search_q = "How many users are in the system?"
    sql      = "SELECT COUNT(*) FROM users"

    configs = [
        dict(enable_quantization=False,  quantization_type="scalar",  label="no quantization"),
        dict(enable_quantization=True,   quantization_type="scalar",  label="scalar INT8 (default)"),
        dict(enable_quantization=True,   quantization_type="binary",  label="binary (dim<512: falls back to scalar)"),
    ]

    print(f"  {'Config':<42s}  {'Strategy':<20s}  {'Confidence':>10s}")
    print("  " + "-" * 78)

    for cfg in configs:
        label = cfg.pop("label")
        settings = Settings(
            backend_type="qdrant",
            qdrant_mode="memory",
            **cfg,
        )
        async with Medha(f"quant_test", embedder=embedder, settings=settings) as medha:
            await medha.store(stored_q, sql)
            hit = await medha.search(search_q)
            conf = f"{hit.confidence:.4f}" if hit.confidence else "   n/a"
            print(f"  {label:<42s}  {hit.strategy.value:<20s}  {conf:>10s}")

## Cell 8: Quantization Search Params

Three settings control how quantized vectors are used **at query time**:

| Setting | Default | Effect |
|---|---|---|
| `quantization_rescore` | `True` | Re-score top candidates with original vectors (better accuracy) |
| `quantization_oversampling` | `None` | Fetch `limit × N` candidates before re-scoring. `None` = Qdrant default |
| `quantization_ignore` | `False` | Skip quantized index entirely; use original vectors (slowest, most accurate) |

**Recommended for Medha:** defaults work well. Lower `rescore` only if original
vectors are on slow disk storage (`on_disk=True`) and latency matters more than precision.

In [ ]:
if not HAS_QDRANT:
    print("Skipping — qdrant-client not installed.")
else:
    stored_q = "How many users are there?"
    search_q = "How many users are in the system?"
    sql      = "SELECT COUNT(*) FROM users"

    configs = [
        dict(rescore=True,  oversampling=None, ignore=False, label="default (rescore=True)"),
        dict(rescore=True,  oversampling=2.0,  ignore=False, label="rescore + oversampling=2.0"),
        dict(rescore=False, oversampling=None, ignore=False, label="rescore=False (quantized only)"),
        dict(rescore=False, oversampling=None, ignore=True,  label="ignore=True (original vectors)"),
    ]

    print(f"  {'Config':<38s}  {'Strategy':<20s}  {'Conf':>6s}  {'Latency':>8s}")
    print("  " + "-" * 80)

    for cfg in configs:
        label = cfg.pop("label")
        settings = Settings(
            backend_type="qdrant",
            qdrant_mode="memory",
            quantization_rescore=cfg["rescore"],
            quantization_oversampling=cfg["oversampling"],
            quantization_ignore=cfg["ignore"],
        )
        async with Medha("qsearch_test", embedder=embedder, settings=settings) as medha:
            await medha.store(stored_q, sql)
            t0 = time.perf_counter()
            hit = await medha.search(search_q)
            latency = (time.perf_counter() - t0) * 1000
            conf = f"{hit.confidence:.4f}" if hit.confidence else "  n/a"
            print(f"  {label:<38s}  {hit.strategy.value:<20s}  {conf}  {latency:>6.1f}ms")

## Cell 9: Hybrid Storage — `on_disk=True`

For large collections that exceed RAM, Qdrant can store **original vectors on disk**
while keeping **quantized vectors in RAM**. This is the hybrid storage pattern:

```python
Settings(
    backend_type="qdrant",
    qdrant_mode="docker",
    on_disk=True,                    # original float32 vectors → disk
    enable_quantization=True,        # create quantized index
    quantization_always_ram=True,    # quantized vectors → RAM  (default)
)
```

| Storage | `on_disk` | `quantization_always_ram` | Vectors in RAM | Vectors on disk |
|---|---|---|---|---|
| Full RAM | False | True | float32 + quantized | nothing |
| **Hybrid (recommended)** | **True** | **True** | **quantized only** | **float32** |
| Full disk | True | False | nothing | float32 + quantized |

`on_disk=True` in memory mode is silently ignored by Qdrant (no persistent storage).

In [ ]:
if not HAS_QDRANT:
    print("Skipping — qdrant-client not installed.")
else:
    # Demonstrate settings construction — on_disk has no observable effect in memory mode
    configs = [
        dict(on_disk=False, quantization_always_ram=True,  label="Full RAM (default)"),
        dict(on_disk=True,  quantization_always_ram=True,  label="Hybrid: original on disk, quantized in RAM"),
        dict(on_disk=True,  quantization_always_ram=False, label="Full disk (slow search)"),
    ]

    stored_q = "How many users are there?"
    search_q = "Total user count"
    sql      = "SELECT COUNT(*) FROM users"

    print(f"  {'Config':<45s}  {'Strategy':<20s}  {'Conf':>6s}")
    print("  " + "-" * 76)

    for cfg in configs:
        label = cfg.pop("label")
        settings = Settings(
            backend_type="qdrant",
            qdrant_mode="memory",
            **cfg,
        )
        async with Medha("ondisk_test", embedder=embedder, settings=settings) as medha:
            await medha.store(stored_q, sql)
            hit = await medha.search(search_q)
            conf = f"{hit.confidence:.4f}" if hit.confidence else "  n/a"
            print(f"  {label:<45s}  {hit.strategy.value:<20s}  {conf}")

    print("\nNote: on_disk has no observable effect in memory mode — use docker/cloud to see it.")

## Cell 10: Multi-Tenant Isolation

Each `Medha` instance creates its own Qdrant collection. Multiple tenants share
the same Qdrant server while staying fully isolated — no data leakage between
collections.

```
qdrant_demo_tenant_acme
qdrant_demo_tenant_globex
qdrant_demo_tenant_initech
```

In [ ]:
if not HAS_QDRANT:
    print("Skipping — qdrant-client not installed.")
else:
    TENANTS = {
        "tenant_acme":   "SELECT COUNT(*) FROM acme_users",
        "tenant_globex": "SELECT COUNT(*) FROM globex_accounts",
        "tenant_initech": "SELECT COUNT(*) FROM initech_employees",
    }

    shared_settings = Settings(backend_type="qdrant", qdrant_mode="memory")

    instances: dict[str, Medha] = {}
    for tenant_id, sql in TENANTS.items():
        m = Medha(
            collection_name=tenant_id,
            embedder=embedder,
            settings=shared_settings,
        )
        await m.start()
        await m.store("How many users?", sql)
        instances[tenant_id] = m
        print(f"Tenant '{tenant_id}' initialized → SQL: {sql}")

    print()

    # Same question, different SQL per tenant — fully isolated collections
    for tenant_id, m in instances.items():
        hit = await m.search("User count")
        print(f"  [{tenant_id}]: {hit.generated_query}")

    for m in instances.values():
        await m.close()

## Cell 11: TTL + `expire()`

TTL is enforced via a Qdrant `DatetimeRange` filter on the `expires_at` payload field.
Active searches automatically exclude expired entries. `expire()` permanently deletes
them by calling `scroll` → `delete`.

In [ ]:
if not HAS_QDRANT:
    print("Skipping — qdrant-client not installed.")
else:
    settings = Settings(backend_type="qdrant", qdrant_mode="memory")

    async with Medha("qdrant_ttl", embedder=embedder, settings=settings) as medha:
        await medha.store(
            "Today's revenue",
            "SELECT SUM(amount) FROM orders WHERE DATE(created_at) = CURDATE()",
            ttl=3600,   # expires in 1 hour
        )
        await medha.store(
            "Expiring soon",
            "SELECT NOW()",
            ttl=1,      # expires immediately
        )

        count_before = await medha._backend.count("qdrant_ttl")
        print(f"Count before sleep  : {count_before}")

        await asyncio.sleep(2)

        # Search skips expired entries via DatetimeRange filter
        hit = await medha.search("Expiring soon")
        print(f"Search 'Expiring soon' after TTL: strategy={hit.strategy.value}")

        # expire() removes them permanently from Qdrant
        removed = await medha.expire()
        count_after = await medha._backend.count("qdrant_ttl")
        print(f"Removed {removed} expired entries via DatetimeRange filter")
        print(f"Count after expire  : {count_after}")

## Cell 12: `stats()` + `dedup_collection()`

In [ ]:
if not HAS_QDRANT:
    print("Skipping — qdrant-client not installed.")
else:
    settings = Settings(
        backend_type="qdrant",
        qdrant_mode="memory",
        collect_stats=True,
    )

    async with Medha("qdrant_dedup", embedder=embedder, settings=settings) as medha:
        # Store the same entry twice — same query_hash, creates duplicates
        for _ in range(2):
            await medha.store("User count", "SELECT COUNT(*) FROM users")
        await medha.store("Active users", "SELECT * FROM users WHERE active = true")

        count_before = await medha._backend.count("qdrant_dedup")
        print(f"Count before dedup : {count_before}")

        dupes = await medha.dedup_collection()
        count_after = await medha._backend.count("qdrant_dedup")
        print(f"Duplicates removed  : {dupes}")
        print(f"Count after dedup   : {count_after}")

        # Stats
        await medha.search("How many users?")
        await medha.search("Something unrelated")
        stats = await medha.stats()
        print(f"\nStats: requests={stats.total_requests}  hit_rate={stats.hit_rate:.2%}  backend_count={stats.backend_count}")

## Cell 13: Cloud Mode — Qdrant Cloud (Placeholder)

`qdrant_mode="cloud"` connects to [Qdrant Cloud](https://cloud.qdrant.io) using a
cluster URL and API key. The Medha API is identical — only the Settings change.

```python
Settings(
    backend_type="qdrant",
    qdrant_mode="cloud",
    qdrant_url="https://your-cluster.us-east4-0.gcp.cloud.qdrant.io",
    qdrant_api_key="your-api-key",
)
```

**Environment variables:**
```bash
export MEDHA_QDRANT_MODE=cloud
export MEDHA_QDRANT_URL=https://your-cluster.us-east4-0.gcp.cloud.qdrant.io
export MEDHA_QDRANT_API_KEY=your-api-key
```

In [ ]:
# Placeholder — requires Qdrant Cloud credentials
settings_cloud = Settings(
    backend_type="qdrant",
    qdrant_mode="cloud",
    qdrant_url="https://your-cluster.us-east4-0.gcp.cloud.qdrant.io",
    qdrant_api_key="your-api-key",
)

print(f"qdrant_mode : {settings_cloud.qdrant_mode}")
print(f"qdrant_url  : {settings_cloud.qdrant_url}")
print("api_key     : [set]" if settings_cloud.qdrant_api_key else "api_key     : [not set]")
print("(Not connecting — placeholder values)")

## Cell 14: Graceful Degradation — Fallback to InMemoryBackend

If `qdrant-client` is not installed, importing `QdrantBackend` raises `ImportError`.
Catch it at startup to fall back transparently to `InMemoryBackend`:

In [ ]:
from medha import InMemoryBackend

def build_backend(settings: Settings):
    """Try QdrantBackend; fall back to InMemoryBackend if qdrant-client is not installed."""
    if settings.backend_type == "qdrant":
        try:
            from medha import QdrantBackend
            return QdrantBackend(settings)
        except ImportError as e:
            print(f"QdrantBackend unavailable ({e}), falling back to InMemoryBackend")
            return InMemoryBackend()
    return None  # let Medha build the backend from settings


settings = Settings(backend_type="qdrant", qdrant_mode="memory")
backend = build_backend(settings)
print(f"Backend selected: {type(backend).__name__}")

medha = Medha(
    "fallback_demo",
    embedder=embedder,
    backend=backend,
    settings=Settings(),
)
await medha.start()
await medha.store("Count users", "SELECT COUNT(*) FROM users")
hit = await medha.search("User count")
print(f"Search: strategy={hit.strategy.value}, query={hit.generated_query!r}")
await medha.close()

## Cell 15: Cleanup — `drop_collection()`

Unlike other backends, Qdrant exposes `drop_collection()` directly on the backend.
This deletes the collection and all its data permanently.

In docker/cloud mode, you can also drop collections from the Qdrant dashboard
or via the REST API:
```bash
curl -X DELETE http://localhost:6333/collections/my_collection
```

In [ ]:
if not HAS_QDRANT:
    print("Skipping — qdrant-client not installed.")
else:
    settings = Settings(backend_type="qdrant", qdrant_mode="memory")

    medha = Medha("cleanup_demo", embedder=embedder, settings=settings)
    await medha.start()

    await medha.store("How many users?", "SELECT COUNT(*) FROM users")
    count = await medha._backend.count("cleanup_demo")
    print(f"Count before drop: {count}")

    await medha._backend.drop_collection("cleanup_demo")
    print("Collection dropped.")

    await medha.close()

## Summary

| Feature | Detail |
|---|---|
| Deployment modes | `memory` (in-process), `docker` (local/remote), `cloud` (Qdrant Cloud) |
| Connection | `qdrant_url` (full URL) or `qdrant_host` + `qdrant_port` |
| Index type | HNSW; tunable via `hnsw_m` + `hnsw_ef_construct` |
| Quantization | Scalar INT8 (default, any dim) or Binary (dim ≥ 512) |
| Search accuracy | `quantization_rescore=True` + optional `quantization_oversampling` |
| Hybrid storage | `on_disk=True` + `quantization_always_ram=True` → originals on disk, quantized in RAM |
| Payload indexes | `template_id`, `query_hash` (KEYWORD), `usage_count` (INTEGER), `normalized_question` (TEXT) |
| TTL | `DatetimeRange` filter on `expires_at` — active searches skip expired entries |
| Persistence | Full — docker/cloud data survives restarts |
| Multi-tenant | One Qdrant collection per Medha instance; all share the same server |
| Cleanup | `drop_collection()` on the backend or DELETE via Qdrant REST API |

**When to choose `QdrantBackend`:**
- You need production-grade **HNSW** with quantization and payload filtering
- Your collection will grow beyond 100k entries
- You want **Qdrant Cloud** for managed, serverless vector search
- You need **hybrid storage** (large collections that exceed RAM)

**Medha 0.3.0 backend overview** — choose based on your infrastructure:

| `backend_type` | Extra deps | Persistence | Best for |
|---|---|---|---|
| `"memory"` | none | no | **Default**; tests, CI, zero-infra demos |
| `"qdrant"` | `medha-archai[qdrant]` | yes | **Production vector search, HNSW + quantization** |
| `"pgvector"` | `medha-archai[pgvector]` | yes | Teams already on PostgreSQL |
| `"elasticsearch"` | `medha-archai[elasticsearch]` | yes | Teams on Elastic stack |
| `"vectorchord"` | `medha-archai[vectorchord]` | yes | High-throughput PostgreSQL |
| `"chroma"` | `medha-archai[chroma]` | yes | Lightweight local store |
| `"weaviate"` | `medha-archai[weaviate]` | yes | Managed cloud vector search |
| `"redis"` | `medha-archai[redis]` | yes | Redis-stack teams |
| `"azure-search"` | `medha-archai[azure-search]` | yes | Azure-native teams |
| `"lancedb"` | `medha-archai[lancedb]` | yes | Zero-infra embedded + cloud object storage |
